# 🎓 Masterclass ELK : Architecture, Ingestion & Recherche (Édition Définitive)
---
Ce classeur Jupyter n'est pas qu'un simple script, c'est **un cours interactif** sur la Stack Elastic, calqué sur les méthodes universitaires de votre professeur.
Nous allons aborder chaque composant et exécuter de **poussées de vraies requêtes de science des données** (plus de 14 cas d'usages complexes) sur Elasticsearch.

*Rappel : Appuyez sur `Shift + Entrée` pour exécuter un bloc gris.*

---
## CHAPITRE 1 : Setup Initial
L'adresse de notre base n'est plus *localhost* mais le vrai nom du conteneur : **`http://elasticsearch:9200`**.

In [ ]:
import requests
from IPython.display import JSON, display

ES_URL = 'http://elasticsearch:9200'
print("✅ Librairies prêtes avec affichage JSON interactif !")

---
## CHAPITRE 2 : Le Laboratoire de Requêtes (Query DSL)
Voici notre **cahier d'exercices analytiques comportant 14 requêtes différentes et avancées (dont 5+ requêtes complexes de type BOOL)**.

### Ex 1. `match_phrase` (Recherche Phrase Exacte)
Le `match` classique coupe votre phrase en morceaux. Le `match_phrase` oblige Elastic à trouver exactement l'expression !

In [ ]:
res = requests.post(f'{ES_URL}/movies_clean/_search', json={
  "size": 2, "query": { "match_phrase": { "overview": "save the world" } }
}).json()

print("Films contenant exactement 'save the world' dans leur synopsis:")
display(JSON(res))

### Ex 2. `prefix` (Recherche par préfixe/auto-complétion)
Idéal pour coder l'auto-complétion d'une barre de recherche ! On trouve les films dont le titre commence par 'inter'

In [ ]:
res = requests.post(f'{ES_URL}/movies_clean/_search', json={
  "size": 2, "query": { "prefix": { "title": "inter" } }
}).json()

print("Autocomplétion sur la racine 'inter' :")
display(JSON(res))

### Ex 3. `wildcard` (Les Jokers / Regex)
Permet de chercher des motifs textuels au milieu d'un mot en utilisant `*`.

In [ ]:
res = requests.post(f'{ES_URL}/movies_clean/_search', json={
  "size": 2, "query": { "wildcard": { "title": "*potter*" } }
}).json()

print("Titre avec astérisques (*potter*) :")
display(JSON(res))

### Ex 4. `bool` (1) : `must_not` + `exists` (Audit Qualité)
Tès pratique pour trouver les lignes corrompues qui n'ont pas de synopsis.

In [ ]:
res = requests.post(f'{ES_URL}/movies_clean/_search', json={
  "size": 2, "query": { "bool": { "must_not": [ { "exists": { "field": "overview" } } ] } }
}).json()

print("Films sans synopsis (Erreur de données) :")
display(JSON(res))

### Ex 5. `terms` Array (Choix Multiples Exacts)
Si l'utilisateur coche plusieurs cases de filtres.

In [ ]:
res = requests.post(f'{ES_URL}/movies_clean/_search', json={
  "size": 2, "query": { "terms": { "original_language": ["ja", "ko", "zh"] } }
}).json()

print("Panel de films d'Asie (Japon, Corée, Chine) :")
display(JSON(res))

### Ex 6. `range` avec expressions Date-Math
L'avantage du NOSQL c'est de manipuler le temps dynamiquement : `now-20y`.

In [ ]:
res = requests.post(f'{ES_URL}/movies_clean/_search', json={
  "size": 2, "query": { "range": { "release_date_ts": { "lt": "now-20y" } } }
}).json()

print("Trouver les oeuvres qui ont plus de 20 ans :")
display(JSON(res))

### Ex 7. Le Scoring : `function_score` (La Magie de l'Ordre)
Croiser la popularité au ratio de recherche textuelle.

In [ ]:
res = requests.post(f'{ES_URL}/movies_clean/_search', json={
  "size": 2, "query": { "function_score": { "query": { "match": { "overview": "space" } }, "field_value_factor": { "field": "popularity", "modifier": "log1p", "factor": 1.2 } } }
}).json()

print("Boost du Score Algorithmique pour les blockbusters Spatiaux :")
display(JSON(res))

### Ex 8. `fuzziness` combiné (L'Auto-correction Avancée)
On autorise des erreurs de frappe (distance de Levenshtein de 2 pour `matrx`).

In [ ]:
res = requests.post(f'{ES_URL}/movies_clean/_search', json={
  "size": 2, "query": { "multi_match": { "query": "matrx", "fields": ["title"], "fuzziness": 2 } }
}).json()

print("Fuzziness Tolérance +2 :")
display(JSON(res))

### Ex 9. `stats` (Statistiques Globales de Data Science)
Moyenne, Médiane, Min et Max en une seule requête.

In [ ]:
res = requests.post(f'{ES_URL}/movies_clean/_search', json={
  "size": 0, "aggs": { "calculs_notes": { "stats": { "field": "vote_average" } } }
}).json()

print("Résumé Descriptif Mathématiques (Aggrégation Statistique) :")
display(JSON(res.get('aggregations',{})))

### Ex 10. Histogramme par Paliers Personnalisés (`histogram`)
Répartition des films par blocs de 2 points de notation.

In [ ]:
res = requests.post(f'{ES_URL}/movies_clean/_search', json={
  "size": 0, "aggs": { "repartition_qualite": { "histogram": { "field": "vote_average", "interval": 2 } } }
}).json()

print("L'Histogramme des notes distribuées par intervalle de 2 points :")
display(JSON(res.get('aggregations',{})))

### Ex 11. `bool` (2) : `must` + `filter` (Précision Chirurgicale)
On cherche des films avec 'Marvel' dans le titre, mais on **filtre** strictement sur ceux ayant une note > 7.

In [ ]:
res = requests.post(f'{ES_URL}/movies_clean/_search', json={
  "size": 2,
  "query": {
    "bool": {
      "must": [ { "match": { "title": "Marvel" } } ],
      "filter": [ { "range": { "vote_average": { "gt": 7 } } } ]
    }
  }
}).json()

print("Films Marvel bien notés (Must + Filter) :")
display(JSON(res))

### Ex 12. `bool` (3) : `should` + `minimum_should_match`
On veut des films qui parlent de 'Hero' OU de 'Justice'. On exige qu'au moins 1 des conditions soit remplie.

In [ ]:
res = requests.post(f'{ES_URL}/movies_clean/_search', json={
  "size": 2,
  "query": {
    "bool": {
      "should": [
        { "match": { "overview": "Hero" } },
        { "match": { "overview": "Justice" } }
      ],
      "minimum_should_match": 1
    }
  }
}).json()

print("Films avec Hero ou Justice (Should) :")
display(JSON(res))

### Ex 13. `bool` (4) : `must` + `must_not` (Exclusion active)
On cherche des films français (`fr`) mais on veut absolument **exclure** ceux qui parlent d''Amour' dans le synopsis.

In [ ]:
res = requests.post(f'{ES_URL}/movies_clean/_search', json={
  "size": 2,
  "query": {
    "bool": {
      "must": [ { "term": { "original_language": "fr" } } ],
      "must_not": [ { "match": { "overview": "Amour" } } ]
    }
  }
}).json()

print("Films Français sans 'Amour' (Must + Must Not) :")
display(JSON(res))

### Ex 14. `bool` (5) : Le Trio Infernal (Must + Should + Filter)
Requête Expert : Films populaires (> 500), boost si Anglais, mais Note >= 5/10.

In [ ]:
res = requests.post(f'{ES_URL}/movies_clean/_search', json={
  "size": 2,
  "query": {
    "bool": {
      "must": [ { "range": { "popularity": { "gt": 500 } } } ],
      "should": [ { "term": { "original_language": "en" } } ],
      "filter": [ { "range": { "vote_average": { "gte": 5 } } } ]
    }
  }
}).json()

print("Requête Combinée (Must + Should + Filter) :")
display(JSON(res))